In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [3]:
import pandas as pd

df = pd.read_csv("news_summary_more.csv")
df.head(2)

,headlines,text
0,upGrad learner switches to career in ML & Al w...,"Saurav Kant, an alumnus of upGrad and IIIT-B's..."
1,Delhi techie wins free food from Swiggy for on...,Kunal Shah's credit card bill payment platform...


In [4]:
len(df)

98401

In [5]:
df.head()

,headlines,text
0,upGrad learner switches to career in ML & Al w...,"Saurav Kant, an alumnus of upGrad and IIIT-B's..."
1,Delhi techie wins free food from Swiggy for on...,Kunal Shah's credit card bill payment platform...
2,New Zealand end Rohit Sharma-led India's 12-ma...,New Zealand defeated India by 8 wickets in the...
3,Aegon life iTerm insurance plan helps customer...,"With Aegon Life iTerm Insurance plan, customer..."
4,"Have known Hirani for yrs, what if MeToo claim...",Speaking about the sexual harassment allegatio...


In [6]:
avg_count = [len(x.split()) for x in df.headlines]
sum(avg_count)/len(avg_count)

9.55387648499507

In [7]:
avg_count = [len(x.split()) for x in df.text]
sum(avg_count)/len(avg_count)

58.23770083637361

In [8]:
!pip install tokenizers

In [9]:
import re
import html

def preprocess_text(text):

    text = html.unescape(text)
    url_pattern = r"https?://\S+|www\.\S+"
    text = re.sub(url_pattern, " <url> ", text)

    text = text.lower()

    text = re.sub(r"[‘’´`]", "'", text)
    text = re.sub(r'[“”]', '"', text)
    text = re.sub(r'[—–]', '-', text)

    text = text.replace("<url>", "PLACEHOLDERURL")

    text = re.sub(r"([!\"#$%&'()*+,-./:;<=>?@[\\\]^_`{|}~])", r" \1 ", text)

    text = text.replace("PLACEHOLDERURL", " <url> ")

    text = re.sub(r"\s+", " ", text).strip()

    return text



df.headlines = df.headlines.apply(lambda x: preprocess_text(x))
df.text = df.text.apply(lambda x: preprocess_text(x))
df.head()

,headlines,text
0,upgrad learner switches to career in ml & al w...,"saurav kant , an alumnus of upgrad and iiit - ..."
1,delhi techie wins free food from swiggy for on...,kunal shah ' s credit card bill payment platfo...
2,new zealand end rohit sharma - led india ' s 1...,new zealand defeated india by 8 wickets in the...
3,aegon life iterm insurance plan helps customer...,"with aegon life iterm insurance plan , custome..."
4,"have known hirani for yrs , what if metoo clai...",speaking about the sexual harassment allegatio...


In [ ]:
df.headlines

,headlines
0,upgrad learner switches to career in ml & al w...
1,delhi techie wins free food from swiggy for on...
2,new zealand end rohit sharma - led india ' s 1...
3,aegon life iterm insurance plan helps customer...
4,"have known hirani for yrs , what if metoo clai..."
...,...
98396,crpf jawan axed to death by maoists in chhatti...
98397,first song from sonakshi sinha ' s ' noor ' ti...
98398,' the matrix ' film to get a reboot : reports
98399,snoop dogg aims gun at clown dressed as trump ...


In [10]:
from tokenizers import ByteLevelBPETokenizer
import os
from tokenizers.processors import TemplateProcessing

os.makedirs("my_bpe_tokenizer", exist_ok=True)

with open("corpus.txt", "w", encoding="utf-8") as f:
    for text in df['text']:
        f.write(text + "\n")
    for headline in df['headlines']:
        f.write(headline + "\n")

bpe_tokenizer = ByteLevelBPETokenizer()

bpe_tokenizer.train(
    files=["corpus.txt"],
    vocab_size=15000,
    min_frequency=2,
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<url>"]
)

bpe_tokenizer.save_model("my_bpe_tokenizer")
print("BPE Tokenizer successfully trained and saved!")


pad_id = bpe_tokenizer.token_to_id("<pad>")
sos_id = bpe_tokenizer.token_to_id("<s>")
eos_id = bpe_tokenizer.token_to_id("</s>")

bpe_tokenizer.post_processor = TemplateProcessing(
    single="<s> $A </s>",
    special_tokens=[
        ("<s>", sos_id),
        ("</s>", eos_id),
    ],
)

bpe_tokenizer.enable_truncation(max_length=32)
bpe_tokenizer.enable_padding(
    length=32,
    pad_id=pad_id,
    pad_token="<pad>"
)

headlines_tok = bpe_tokenizer.encode_batch(df['headlines'].tolist())

bpe_tokenizer.enable_truncation(max_length=128)
bpe_tokenizer.enable_padding(
    length=128,
    pad_id=pad_id,
    pad_token="<pad>"
)

text_tok = bpe_tokenizer.encode_batch(df['text'].tolist())

BPE Tokenizer successfully trained and saved!


In [ ]:
headlines_tok = [headlines_tok[i].ids for i in range(len(headlines_tok))]
text_tok = [text_tok[i].ids for i in range(len(text_tok))]

In [ ]:
bpe_tokenizer.decode(headlines_tok[0])

'upgrad learner switches to career in ml & al with 90 % salary hike'

In [ ]:
headlines_tok[:2]

[[0,
  378,
  9752,
  2917,
  1079,
  1003,
  11841,
  294,
  2591,
  284,
  3184,
  1466,
  354,
  375,
  3385,
  651,
  4021,
  4872,
  2,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [0,
  2206,
  1244,
  533,
  3126,
  1628,
  1738,
  407,
  12482,
  324,
  664,
  418,
  315,
  3116,
  2,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]]

In [ ]:
import numpy as np
headlines_tok = np.array(headlines_tok, dtype = np.int16)
text_tok = np.array(text_tok, dtype = np.int16)

In [ ]:
import pickle
data = (text_tok, headlines_tok)
with open('summ_data.pkl', 'wb') as f:
    pickle.dump(data, f)